# GenAI & Agent Fundamentals + AutoGen Basics

## Learning Objectives

By the end of this notebook, learners should be able to:

- Explain what Generative AI, LLMs, tokens, embeddings, prompts, and context windows are.
- Explain what an AI agent is and how it differs from a chatbot.
- Understand core agent components: LLM, tools, memory, planning, and feedback.
- Build a simple rule-based agent in Python.
- Build a simple tool-using agent in Python.
- Understand what AutoGen is and where it fits in the agent ecosystem.
- Install AutoGen AgentChat.
- Create a basic AutoGen `AssistantAgent`.
- Understand how to connect AutoGen with OpenAI or Azure OpenAI.

# Module 1: GenAI & Agent Fundamentals

## 1. What is Generative AI?

**Generative AI** is a type of AI that can create new content such as:

- text,
- code,
- images,
- audio,
- video,
- structured data.

In this course, we focus mainly on **text and code generation** using Large Language Models.

### Simple Explanation

Traditional software follows fixed instructions.

Generative AI predicts and generates likely output based on patterns learned from large datasets.

Example:

```text
Input: Write a polite email asking for project status.
Output: Hi Team, Could you please share the latest status...
```

### Key Terms

| Term | Meaning |
|---|---|
| Prompt | Input given to the model |
| Completion / Response | Output generated by the model |
| Token | Small unit of text processed by the model |
| Context Window | Maximum amount of text the model can consider at one time |
| Embedding | Numeric representation of text meaning |
| Hallucination | Confident but incorrect model output |

## 2. AI → ML → Deep Learning → GenAI

A simple view:

```text
Artificial Intelligence
    ↓
Machine Learning
    ↓
Deep Learning
    ↓
Foundation Models
    ↓
Generative AI
    ↓
Agentic AI
```

### Explanation

- **AI**: Broad field of making machines act intelligently.
- **ML**: Systems learn patterns from data.
- **Deep Learning**: Neural networks learn complex patterns.
- **Foundation Models**: Large models trained on broad data.
- **GenAI**: Uses foundation models to generate content.
- **Agentic AI**: Uses LLMs plus tools, memory, planning, and actions.

## 3. What is an LLM?

A **Large Language Model (LLM)** is a model trained to understand and generate human-like text.

Examples:
- GPT models
- Claude models
- Gemini models
- Llama models
- Mistral models

LLMs are useful for:
- summarization,
- question answering,
- code generation,
- test case generation,
- data extraction,
- reasoning assistance,
- automation workflows.

### Important Limitation

An LLM is not a database. It may generate incorrect answers if:
- the prompt is unclear,
- the answer requires private/current data,
- the model does not have enough context,
- the task needs tool access.

## 4. Tokens and Context Window

LLMs process text as **tokens**, not exactly as words.

Approximate example:

```text
"AutoGen helps build AI agents"
```

may become tokens like:

```text
["Auto", "Gen", " helps", " build", " AI", " agents"]
```

### Context Window

The context window is the maximum amount of information the model can consider in one request.

If the conversation or document is too large, you need strategies like:

- summarization,
- chunking,
- retrieval,
- memory,
- RAG.

In [2]:
# Simple token approximation
# This is NOT the exact tokenizer used by LLMs.
# It is only for classroom demonstration.

text = "AutoGen helps build AI agents that can use tools and collaborate."
approx_tokens = text.split()

print("Text:", text)
print("Approx token count:", len(approx_tokens))
print("Approx tokens:", approx_tokens)

Text: AutoGen helps build AI agents that can use tools and collaborate.
Approx token count: 11
Approx tokens: ['AutoGen', 'helps', 'build', 'AI', 'agents', 'that', 'can', 'use', 'tools', 'and', 'collaborate.']


## 5. Prompt Engineering Basics

A **prompt** is the instruction or input given to the LLM.

A good prompt usually contains:

1. **Role** - who the model should act as.
2. **Task** - what it should do.
3. **Context** - background information.
4. **Constraints** - rules or limitations.
5. **Output format** - expected structure.

### Example Prompt Template

```text
You are a senior test automation engineer.
Task: Generate test cases for the login feature.
Context: Users can login using email and password.
Constraints: Include positive, negative, and security test cases.
Output format: Table with columns: Test Case, Steps, Expected Result.
```

In [4]:
def build_prompt(role, task, context, constraints, output_format):
    return f"""
You are {role}.

Task:
{task}

Context:
{context}

Constraints:
{constraints}

Output Format:
{output_format}
""".strip()

prompt = build_prompt(
    role="a senior test automation engineer",
    task="Generate test cases for the login feature.",
    context="Users can login using email and password.",
    constraints="Include positive, negative, boundary, and security cases.",
    output_format="A table with Test Case, Steps, and Expected Result."
)

print(prompt)

You are a senior test automation engineer.

Task:
Generate test cases for the login feature.

Context:
Users can login using email and password.

Constraints:
Include positive, negative, boundary, and security cases.

Output Format:
A table with Test Case, Steps, and Expected Result.


## 6. What is an AI Agent?

An **AI Agent** is a system that can use an LLM to reason, decide, and act toward a goal.

A simple chatbot usually responds to messages.

An agent can:

- understand a goal,
- break the goal into steps,
- use tools,
- remember information,
- ask for clarification,
- execute actions,
- validate results.

### Agent Components

```text
User Goal
   ↓
Agent
   ├── LLM / Reasoning
   ├── Tools
   ├── Memory
   ├── Planning
   └── Feedback / Evaluation
```

## 7. Chatbot vs AI Agent

| Chatbot | AI Agent |
|---|---|
| Mostly responds to user input | Can plan and take actions |
| Usually no tool execution | Can call tools/APIs/functions |
| Limited memory | Can use short-term and long-term memory |
| Conversation-focused | Goal/task-focused |
| Example: FAQ bot | Example: test automation agent generating and executing scripts |

## 8. Simple Rule-Based Agent

Before using LLMs, let us build a simple agent manually.

This agent:
- receives a task,
- identifies the intent,
- calls a tool,
- returns a result.

This helps learners understand the agent concept without requiring an API key.

In [6]:
def calculator_tool(expression):
    """A very small calculator tool for safe classroom examples."""
    allowed_chars = set("0123456789+-*/(). ")
    if not set(expression).issubset(allowed_chars):
        return "Invalid expression. Only basic arithmetic is allowed."
    try:
        return eval(expression, {"__builtins__": {}})
    except Exception as e:
        return f"Error: {e}"

def simple_agent(user_message):
    """Simple rule-based agent."""
    user_message_lower = user_message.lower()

    if "calculate" in user_message_lower:
        expression = user_message_lower.replace("calculate", "").strip()
        result = calculator_tool(expression)
        return f"The calculated result is: {result}"

    if "hello" in user_message_lower or "hi" in user_message_lower:
        return "Hello! I am a simple agent. I can calculate basic arithmetic."

    return "I do not know how to handle this request yet."

print(simple_agent("Hi"))
print(simple_agent("calculate 10 + 25 * 2"))

Hello! I am a simple agent. I can calculate basic arithmetic.
The calculated result is: 60


## 9. Tool-Using Agent

A tool is a function that an agent can call.

Examples of tools:

- calculator,
- weather API,
- database query,
- search engine,
- file reader,
- test execution script,
- ticket creation API.

Below is a small example of an agent that selects the correct tool.

In [8]:
def word_count_tool(text):
    return len(text.split())

def uppercase_tool(text):
    return text.upper()

tools = {
    "word_count": word_count_tool,
    "uppercase": uppercase_tool,
    "calculator": calculator_tool
}

def tool_using_agent(task, tool_name, input_data):
    if tool_name not in tools:
        return f"Tool '{tool_name}' is not available."

    tool = tools[tool_name]
    result = tool(input_data)

    return {
        "task": task,
        "tool_used": tool_name,
        "input": input_data,
        "result": result
    }

tool_using_agent(
    task="Count words in a sentence",
    tool_name="word_count",
    input_data="AutoGen is useful for building multi agent systems"
)

{'task': 'Count words in a sentence',
 'tool_used': 'word_count',
 'input': 'AutoGen is useful for building multi agent systems',
 'result': 8}

## 10. Memory in Agents

Memory helps an agent remember useful information.

Types of memory:

| Memory Type | Meaning | Example |
|---|---|---|
| Short-term memory | Current conversation context | User's current question |
| Long-term memory | Saved facts/preferences | User prefers Python examples |
| Episodic memory | Past events/interactions | Previous support ticket |
| Semantic memory | Knowledge facts | Product documentation |

Below is a very simple memory example.

In [10]:
class SimpleMemoryAgent:
    def __init__(self):
        self.memory = {}

    def remember(self, key, value):
        self.memory[key] = value
        return f"Remembered {key} = {value}"

    def recall(self, key):
        return self.memory.get(key, "I do not remember that.")

agent = SimpleMemoryAgent()

print(agent.remember("preferred_language", "Python"))
print(agent.recall("preferred_language"))
print(agent.recall("favorite_tool"))

Remembered preferred_language = Python
Python
I do not remember that.


## 11. Planning in Agents

Planning means breaking a larger goal into smaller steps.

Example goal:

```text
Create test cases for a login page and generate automation script.
```

Possible plan:

1. Understand login requirements.
2. Identify positive and negative scenarios.
3. Generate test cases.
4. Generate automation script.
5. Review script.
6. Suggest improvements.

In [12]:
def simple_planner(goal):
    if "test" in goal.lower() and "login" in goal.lower():
        return [
            "Understand login requirements",
            "Identify valid and invalid login scenarios",
            "Create functional test cases",
            "Create security test cases",
            "Generate automation script",
            "Review and improve the script"
        ]
    return [
        "Understand the goal",
        "Break the goal into smaller tasks",
        "Execute each task",
        "Validate the final output"
    ]

goal = "Create test cases and automation script for login page"
plan = simple_planner(goal)

for i, step in enumerate(plan, start=1):
    print(f"{i}. {step}")

1. Understand login requirements
2. Identify valid and invalid login scenarios
3. Create functional test cases
4. Create security test cases
5. Generate automation script
6. Review and improve the script


# Module 2: AutoGen Basics

## 12. What is AutoGen?

**AutoGen** is an open-source framework from Microsoft for building AI agents and multi-agent applications.

AutoGen helps you create agents that can:

- talk to users,
- talk to other agents,
- use tools,
- execute workflows,
- collaborate on complex tasks.

The current recommended beginner package is **AutoGen AgentChat**.

### Main AutoGen Packages

| Package | Purpose |
|---|---|
| `autogen-agentchat` | High-level API for creating conversational agents |
| `autogen-core` | Lower-level event-driven framework |
| `autogen-ext` | Extensions such as model clients and integrations |
| `autogenstudio` | GUI / low-code experimentation environment |

Official installation commonly uses:

```bash
pip install -U "autogen-agentchat" "autogen-ext[openai]"
```

## 13. Why AutoGen?

AutoGen is useful when you want to build applications where:

- multiple agents collaborate,
- each agent has a specialized role,
- agents use tools,
- agents solve tasks step by step,
- human approval is needed,
- code or workflow execution is part of the solution.

### Example

```text
User asks: Analyze this production incident.

Planner Agent:
Breaks task into steps.

Log Analyzer Agent:
Reads logs and finds errors.

RCA Agent:
Finds probable root cause.

Resolution Agent:
Suggests fix.

Reviewer Agent:
Validates final answer.
```

## 14. AutoGen Architecture - Beginner View

```text
User
 ↓
AgentChat API
 ↓
AssistantAgent / UserProxyAgent / Team
 ↓
Model Client
 ↓
LLM Provider
```

### Key Concepts

| Concept | Meaning |
|---|---|
| Agent | Entity that can receive messages and produce responses |
| Model Client | Connection to LLM provider |
| Tool | Python function or external capability used by an agent |
| Team | Group of agents working together |
| Message | Communication between user and agent or between agents |

## 15. Install AutoGen

Run this cell only once in your environment.

> Note: This requires internet access from your notebook environment.

In [18]:
# Uncomment and run if AutoGen is not installed.
# !pip install -U "autogen-agentchat" "autogen-ext[openai]" python-dotenv

## 16. Verify AutoGen Installation

In [22]:
try:
    import autogen_agentchat
    import autogen_ext
    print("AutoGen AgentChat is installed successfully.")
except ImportError as e:
    print("AutoGen packages are not installed yet.")
    print("Install using:")
    print('pip install -U "autogen-agentchat" "autogen-ext[openai]" python-dotenv')
    print("Error:", e)

AutoGen AgentChat is installed successfully.


## 17. AutoGen Model Client Setup

To use AutoGen with OpenAI, you typically need:

```python
from autogen_ext.models.openai import OpenAIChatCompletionClient
```

For Azure OpenAI, you typically use:

```python
from autogen_ext.models.openai import AzureOpenAIChatCompletionClient
```

You must set your API key securely using environment variables.

Never hardcode secrets directly in notebooks.

In [28]:
import os
from dotenv import load_dotenv
# Example:
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

api_key_available = bool(os.getenv("OPENAI_API_KEY"))
print("OPENAI_API_KEY available:", api_key_available)

OPENAI_API_KEY available: True


## 18. First AutoGen AssistantAgent

This example creates an AutoGen `AssistantAgent`.

It requires:
- `autogen-agentchat`,
- `autogen-ext[openai]`,
- a valid `OPENAI_API_KEY`.

If you do not have an API key, read the code for understanding and skip execution.

In [34]:
import os
import asyncio

async def run_first_autogen_agent():
    try:
        from autogen_agentchat.agents import AssistantAgent
        from autogen_ext.models.anthropic import AnthropicChatCompletionClient
    except ImportError:
        print("AutoGen or Anthropic extension is not installed.")
        return

    if not os.getenv("ANTHROPIC_API_KEY"):
        print("ANTHROPIC_API_KEY not found. Skipping live AutoGen call.")
        return

    model_client = AnthropicChatCompletionClient(
        model="claude-sonnet-4-20250514",
        api_key=os.getenv("ANTHROPIC_API_KEY")
    )

    assistant = AssistantAgent(
        name="genai_trainer",
        model_client=model_client,
        system_message="You are a helpful GenAI trainer. Explain concepts clearly with examples."
    )

    response = await assistant.run(
        task="Explain AI agents in 5 bullet points."
    )

    print(response)

    await model_client.close()

# In Jupyter
await run_first_autogen_agent()

BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CbeLAofk361WXDT4zDvvq'}

## 19. AutoGen Agent with a Tool

Agents become more powerful when they can use tools.

A tool can be a normal Python function.

Example tool:

```python
def add_numbers(a: int, b: int) -> int:
    return a + b
```

The agent can call the tool when it needs calculation.

In [ ]:
# Optional live AutoGen tool example.
# Requires AutoGen installation and OPENAI_API_KEY.

import os

def add_numbers(a: int, b: int) -> int:
    """Add two integers and return the result."""
    return a + b

async def run_autogen_tool_agent():
    try:
        from autogen_agentchat.agents import AssistantAgent
        from autogen_ext.models.openai import OpenAIChatCompletionClient
    except ImportError:
        print("AutoGen is not installed. Please install it first.")
        return

    if not os.getenv("OPENAI_API_KEY"):
        print("OPENAI_API_KEY not found. Skipping live AutoGen call.")
        return

    model_client = OpenAIChatCompletionClient(
        model="gpt-4o-mini",
        api_key=os.getenv("OPENAI_API_KEY")
    )

    assistant = AssistantAgent(
        name="calculator_agent",
        model_client=model_client,
        tools=[add_numbers],
        system_message="You are a calculator agent. Use tools when needed."
    )

    response = await assistant.run(task="What is 145 + 278? Use the tool.")
    print(response)

    await model_client.close()

# In Jupyter, run:
# await run_autogen_tool_agent()

## 20. AutoGen Without API Key: Mock Agent Demo

The following example does not use AutoGen directly.

It simulates the same concept:
- user task,
- assistant agent,
- response generation.

This is useful for classroom explanation when learners do not have API keys.

In [14]:
class MockAssistantAgent:
    def __init__(self, name, system_message):
        self.name = name
        self.system_message = system_message

    def run(self, task):
        return {
            "agent": self.name,
            "system_message": self.system_message,
            "task": task,
            "response": "This is a mock response. In real AutoGen, an LLM would generate this."
        }

mock_agent = MockAssistantAgent(
    name="genai_trainer",
    system_message="You are a helpful GenAI trainer."
)

mock_agent.run("Explain AI agents in simple terms.")

{'agent': 'genai_trainer',
 'system_message': 'You are a helpful GenAI trainer.',
 'task': 'Explain AI agents in simple terms.',
 'response': 'This is a mock response. In real AutoGen, an LLM would generate this.'}

## 21. AutoGen Basic Flow

A beginner AutoGen application usually follows this flow:

```text
1. Install AutoGen packages
2. Import required classes
3. Configure model client
4. Create agent
5. Add tools if needed
6. Run the agent with a task
7. Display response
8. Close model client
```

### Minimal Pattern

```python
model_client = OpenAIChatCompletionClient(...)
assistant = AssistantAgent(...)
response = await assistant.run(task="...")
await model_client.close()
```

## 22. Common Beginner Errors

| Error | Possible Reason | Fix |
|---|---|---|
| `ModuleNotFoundError: autogen_agentchat` | Package not installed | Install `autogen-agentchat` |
| API key not found | Environment variable missing | Set `OPENAI_API_KEY` |
| Authentication error | Wrong/expired key | Verify key |
| Model not found | Incorrect model name | Use valid model name |
| Await error | Async function used incorrectly | In Jupyter use `await function_name()` |
| Version mismatch | Old AutoGen examples | Use current AgentChat docs |

## 23. Trainer Discussion Questions

Use these questions during class:

1. How is an AI agent different from a chatbot?
2. Why do agents need tools?
3. What can go wrong if an agent has unrestricted tool access?
4. Why is memory useful in agentic systems?
5. When should we use multi-agent systems instead of a single agent?
6. What kind of enterprise use cases are suitable for AutoGen?

## 24. Mini Assignment

Build a simple Python agent that can:

1. Accept a user task.
2. Decide whether to use:
   - calculator tool,
   - word count tool,
   - uppercase tool.
3. Execute the selected tool.
4. Return the final result.

Bonus:
- Add memory to remember the last task.
- Add basic error handling.

In [16]:
# Starter code for mini assignment

def calculator_tool(expression):
    allowed_chars = set("0123456789+-*/(). ")
    if not set(expression).issubset(allowed_chars):
        return "Invalid expression."
    return eval(expression, {"__builtins__": {}})

def word_count_tool(text):
    return len(text.split())

def uppercase_tool(text):
    return text.upper()

class MiniAgent:
    def __init__(self):
        self.last_task = None

    def run(self, user_message):
        self.last_task = user_message
        msg = user_message.lower()

        if msg.startswith("calculate"):
            expression = user_message.replace("calculate", "", 1).strip()
            return calculator_tool(expression)

        if msg.startswith("count words"):
            text = user_message.replace("count words", "", 1).strip()
            return word_count_tool(text)

        if msg.startswith("uppercase"):
            text = user_message.replace("uppercase", "", 1).strip()
            return uppercase_tool(text)

        return "I can calculate, count words, or convert text to uppercase."

agent = MiniAgent()

print(agent.run("calculate 100 + 50 / 2"))
print(agent.run("count words AutoGen is useful for agentic AI"))
print(agent.run("uppercase welcome to genai course"))
print("Last task:", agent.last_task)

125.0
6
WELCOME TO GENAI COURSE
Last task: uppercase welcome to genai course


# Summary

In these two modules, we covered:

## GenAI & Agent Fundamentals

- Generative AI
- LLMs
- Tokens
- Prompt engineering
- AI agents
- Tools
- Memory
- Planning

## AutoGen Basics

- What AutoGen is
- Why AutoGen is useful
- AutoGen packages
- AgentChat basics
- AssistantAgent
- Model clients
- Tool integration
- Mock agent demo

Next recommended modules:

1. Creating Agents
2. Multi-Agent Systems
3. GroupChat & Orchestration

## References for Trainer

- Microsoft AutoGen GitHub repository
- Microsoft AutoGen AgentChat documentation
- PyPI package: `autogen-agentchat`

> AutoGen APIs can change over time. For live corporate training, verify package versions before class.